## Notebook to download human proteome AF2 model and run AFragmenter

### Step 1 - Get uniprot accession from UniProt human proteome (UP000005640)
- AFragmenter CLI entry point requires uniprot accession as an argument to download the latest AF2 model.
- Get uniprot human proteome from this URL, then break uniprot accessions down into subset, which will be read by SLURM array jobs

URL for human proteome:
https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cgene_names%2Corganism_name%2Clength&format=tsv&query=%28%28proteome%3AUP000005640%29%29


In [ ]:
import datetime
import subprocess

# Download human proteome .tsv file
proteome_url = "https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cgene_names%2Corganism_name%2Clength&format=tsv&query=%28%28proteome%3AUP000005640%29%29"
access_date = datetime.datetime.today().strftime('%Y_%m_%d')
proteome_path = f"./data/proteome/uniprotkb_proteome_UP000005640_{access_date}.tsv.gz"

# Use subprocess to call wget properly with arguments as a list
subprocess.run([
    "wget", proteome_url, "-O", proteome_path
], check=True)

# Unzip the downloaded file
subprocess.run([
    "gunzip", proteome_path
], check=True)

proteome_path = proteome_path.replace('.gz', '')

--2025-11-27 09:44:08--  https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cgene_names%2Corganism_name%2Clength&format=tsv&query=%28%28proteome%3AUP000005640%29%29
Resolving rest.uniprot.org (rest.uniprot.org)... 193.62.193.81
Connecting to rest.uniprot.org (rest.uniprot.org)|193.62.193.81|:443... connected.
HTTP request sent, awaiting response... 200 
Length: unspecified [text/plain]
Saving to: ‘./data/proteome/uniprotkb_proteome_UP000005640_2025_11_27.tsv’

     0K .......... .......... .......... .......... .......... 94.4K
    50K .......... .......... .......... .......... .......... 82.0K
   100K .......... .......... .......... .......... ..........  160K
   150K .......... .......... .......... .......... .......... 56.0K
   200K .......... .......... .......... .......... .......... 85.7K
   250K .......... .......... .......... .......... ..........  155K
   300K .......... .......... .......... .......... ..........  1

CompletedProcess(args=['wget', 'https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cgene_names%2Corganism_name%2Clength&format=tsv&query=%28%28proteome%3AUP000005640%29%29', '-O', './data/proteome/uniprotkb_proteome_UP000005640_2025_11_27.tsv'], returncode=0)

In [5]:
# Try to read uniref_identity_0_5_AND_taxonomy_id_2025_11_26.tsv
import pandas as pd

df_proteome = pd.read_csv(proteome_path, sep='\t')

df_proteome.head()


,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length
0,A0A087WVL8,unreviewed,A0A087WVL8_HUMAN,Fragile X messenger ribonucleoprotein 1 (Fragi...,FMR1,Homo sapiens (Human),548
1,A0A087WXI3,unreviewed,A0A087WXI3_HUMAN,Fragile X messenger ribonucleoprotein 1 (Fragi...,FMR1,Homo sapiens (Human),536
2,A0A087WY29,unreviewed,A0A087WY29_HUMAN,Fragile X messenger ribonucleoprotein 1 (Fragi...,FMR1,Homo sapiens (Human),561
3,A0A087WYG2,unreviewed,A0A087WYG2_HUMAN,C-Jun-amino-terminal kinase-interacting protei...,MAPK8IP3,Homo sapiens (Human),1337
4,A0A087WZT3,unreviewed,A0A087WZT3_HUMAN,BOLA2-SMG1P6 readthrough,BOLA2-SMG1P6,Homo sapiens (Human),44


In [ ]:
# Paths to directories written by slurm job
subset_dir = "data/subset"       # dir containing input subset csv files
out_dir = "data/out"               # dir to save domainome .pdb and domains .csv files
domainome_dir = "data/domainome"   # dir to save subset out csv files (for tracking status of each subset)

In [ ]:
# split all rows of df_proteome into 500 subset dfs, and save each subset to a .csv file to data/subset
import os

os.makedirs(subset_dir, exist_ok=True)

num_subsets = 500
for i in range(num_subsets):
    subset_df = df_proteome.iloc[i::num_subsets]
    subset_df.to_csv(f'data/subset/subset_{i}.csv', index=False)


In [ ]:
# Submit the job to the cluster
!sbatch  --array 1 slurm/afdb_to_domain.sh $subset_dir $domainome_dir $out_dir

sbatch: [ESTIMATION] The estimated cost of this job is CHF 0.01
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 1.75        │ 0.05        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 1.65        │ 0.05        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)
Submitted batch job 43772546


### Step 2 - summarize results, and compute postprocessing metrics

Postprocessing metrics from https://github.com/igashov/tricomplex-design/blob/residue_number_mapping/notebooks/database_summary.ipynb


In [18]:
# Read all subset out csv files, and summarize results
import os
import pandas as pd
import glob

# Read all subset out files into a single DataFrame
subset_out_files = glob.glob(os.path.join(out_dir, 'subset_*_out.csv'))
df_subset_out = pd.concat([pd.read_csv(f) for f in subset_out_files])
print(f"df_subset_out.shape: {df_subset_out.shape}")
df_subset_out.head()

df_subset_out.shape: (840, 9)


,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Status,Error
0,A0A087WY29,unreviewed,A0A087WY29_HUMAN,Fragile X messenger ribonucleoprotein 1 (Fragi...,FMR1,Homo sapiens (Human),561,success,NaN
1,O00422,reviewed,SAP18_HUMAN,Histone deacetylase complex subunit SAP18 (18 ...,SAP18 GIG38,Homo sapiens (Human),153,success,NaN
2,O43435,reviewed,TBX1_HUMAN,T-box transcription factor TBX1 (T-box protein...,TBX1,Homo sapiens (Human),398,success,NaN
3,O75581,reviewed,LRP6_HUMAN,Low-density lipoprotein receptor-related prote...,LRP6,Homo sapiens (Human),1613,success,NaN
4,O96011,reviewed,PX11B_HUMAN,Peroxisomal membrane protein 11B (Peroxin-11B)...,PEX11B,Homo sapiens (Human),259,success,NaN


In [19]:
# Read all domain .csv files into a single DataFrame
domain_files = glob.glob(os.path.join(domainome_dir, "*", 'AF-*-model*.csv'))
df_domain = pd.concat([pd.read_csv(f) for f in domain_files])
print(f"df_domain.shape: {df_domain.shape}")
df_domain.head()

df_domain.shape: (2196, 7)


,structure,domain,n_residues,resi_start,resi_end,mean_pae,sequence
0,AF-A0A3B3IU53-F1-model_v6.pdb,-1,22,1,22,11.882775,MKMMKTKEEPELQTRREMEERT
1,AF-A0A3B3IU53-F1-model_v6.pdb,1,48,23,70,3.351020,ITIEIPEVLKKQLEDDCYYINRRKRLVKLPCQTNIITILESYVKHFAI
0,AF-Q13002-F1-model_v6.pdb,1,33,1,33,11.394130,MKIIFPILSNPVFRRTVKLLLCLLWIGYSQGTT
1,AF-Q13002-F1-model_v6.pdb,2,385,34,418,6.971705,HVLRFGGIFEYVESGPMGAEELAFRFAVNTINRNRTLLPNTTLTYD...
2,AF-Q13002-F1-model_v6.pdb,-1,10,419,428,7.931507,GKPANITDSL


In [ ]:
# compare this domain sequence with the domain sequence in Shuhao's domainome
df_old_domainome_info = pd.read_csv("/work/upthomae/Meng/filtered_domainome/human_domainome_info_cleaned_YM.csv")
df_old_domainome_info.iloc[0]

id                                                   A0A024R1R8-F1-dom-01_A
uniprot_accn                                                     A0A024R1R8
gene_name                                                             TMA7B
protein_name                    Translation machinery-associated protein 7B
sequence                         [ACE]QAKEMDEEEKAFKQKQKEEQKKLEVLKAKVVG[NME]
resi                      17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,3...
full_sequence             MSSHEGGKKKALKQPKKQAKEMDEEEKAFKQKQKEEQKKLEVLKAK...
deeptmhmm_annotation                     ?IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII?
mdtraj_dssp_annotation                   ?CHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHC?
plddt                     62.95,64.14,67.46,69.75,70.33,74.68,82.96,82.7...
num_residues                                                             34
geodesic_abs_length                                               58.115369
geodesic_norm_length                                               3.886293
drug_target_

In [81]:
# ----- Function to visualize and compare domain boundaries -----

import py3Dmol

def show_domain_comparison_pdb(
    pdb_path,
    df_old_domains,
    df_new_domains,
    width=800,
    height=400
):
    """
    Create side-by-side py3Dmol views of old vs new domain definitions,
    with explicit titles for each view.

    Parameters
    ----------
    pdb_path : str or list
        Full path to a .pdb file (string), or an SList (from !find or !ls shell commands).
        Assumed to match the residue numbering of the domain dataframes (1-based).
    df_old_domains : pandas.DataFrame
        Must contain columns ['resi_start', 'resi_end'] (1-based, inclusive).
    df_new_domains : pandas.DataFrame
        Must contain columns ['resi_start', 'resi_end'] (1-based, inclusive).
    width, height : int
        Size of the py3Dmol canvas.
    """
    # If given a shell command result (SList), get the string path
    if hasattr(pdb_path, '__class__') and pdb_path.__class__.__name__ == "SList":
        pdb_files = [str(x).strip() for x in pdb_path if str(x).strip()]
        if not pdb_files:
            raise ValueError("No PDB files found in SList for pdb_path.")
        pdb_path_str = pdb_files[0]
    elif isinstance(pdb_path, list):
        pdb_files = [str(x).strip() for x in pdb_path if str(x).strip()]
        if not pdb_files:
            raise ValueError("No PDB files found in provided pdb_path list.")
        pdb_path_str = pdb_files[0]
    else:
        pdb_path_str = str(pdb_path)

    # Read PDB as text
    with open(pdb_path_str, 'r') as f:
        pdb_str = f.read()

    # Color palette for domains (will wrap around if more domains)
    palette = [
        'red', 'orange', 'yellow', 'green', 'cyan',
        'blue', 'magenta', 'purple', 'salmon', 'lime'
    ]

    # 1x2 grid: left = old, right = new
    view = py3Dmol.view(viewergrid=(1, 2), width=width, height=height)

    # Helper to set styles for one panel
    def _apply_domains(df_domains, grid_pos):
        # Add the model in this panel
        view.addModel(pdb_str, 'pdb', viewer=grid_pos)

        # Default: everything grey
        view.setStyle(
            {'cartoon': {'color': 'lightgrey'}},
            viewer=grid_pos
        )

        # Color each domain
        for i, row in df_domains.iterrows():
            start = int(row['resi_start'])
            end = int(row['resi_end'])
            color = palette[i % len(palette)]

            # Residue selection: assumes PDB uses 1-based residue numbering
            selection = {'resi': f'{start}-{end}'}
            view.setStyle(
                selection,
                {'cartoon': {'color': color}},
                viewer=grid_pos
            )

        # Nicely center this panel
        view.zoomTo(viewer=grid_pos)

    # Left: old domains
    _apply_domains(df_old_domains, grid_pos=(0, 0))

    # Right: new domains
    _apply_domains(df_new_domains, grid_pos=(0, 1))

    # Add titles to each visualization with smaller font and transparent background
    # (py3Dmol has no direct 'title' method, but we can use addLabel for display)
    view.addLabel(
        "OLD",
        {
            "fontColor": "black",
            "backgroundColor": "rgba(0,0,0,0)",  # transparent background
            "borderColor": "rgba(0,0,0,0)",      # transparent border
            "inFront": True,
            "showBackground": False,             # force no background
            "fontSize": 12,                      # smaller font
            "alignment": "topLeft"
        },
        viewer=(0, 0)
    )
    view.addLabel(
        "NEW",
        {
            "fontColor": "black",
            "backgroundColor": "rgba(0,0,0,0)",  # transparent background
            "borderColor": "rgba(0,0,0,0)",      # transparent border
            "inFront": True,
            "showBackground": False,
            "fontSize": 12,
            "alignment": "topLeft"
        },
        viewer=(0, 1)
    )

    return view


def prepare_and_show_domain_comparison(
    uniprot_accn,
    df_old_domainome_info,
    df_domain,
    pdb_dir="/scratch/ymeng/splitdomain/AFragmenter/data/domainome/"
):
    """
    Prepares domain boundaries and visualizes comparison for a given UniProt accession.

    Parameters
    ----------
    uniprot_accn : str
        UniProt accession id, e.g., 'Q13002'
    df_old_domainome_info : pd.DataFrame
        DataFrame with the legacy domain information for all proteins.
    df_domain : pd.DataFrame
        DataFrame with the new domain boundary information for all proteins.
    pdb_dir : str
        Directory path where the PDB files are located.

    Returns
    -------
    view : py3Dmol.view
        3DMol visualization object comparing old and new domain boundaries.
    df_old_domain_boundaries : pd.DataFrame
        Old domain boundaries dataframe for this UniProt accession.
    df_new_domain_boundaries : pd.DataFrame
        New domain boundaries dataframe for this UniProt accession.
    full_sequence : str
        Full sequence from the old domainome info for this UniProt accession.
    pdb_path : str or SList
        Path(s) to the matching PDB file(s).
    """
    import re
    import pandas as pd

    # Helper function to return a dataframe of domain boundaries for a uniprot_accn
    def get_domain_boundaries(uniprot_accn, df_old_domainome_info):
        df_uniprot = df_old_domainome_info[df_old_domainome_info['uniprot_accn'] == uniprot_accn]
        rows = []
        for _, row in df_uniprot.iterrows():
            resi = row['resi']
            sequence = row['sequence']
            # Remove anything enclosed in square brackets (e.g., [ACE], [NME], etc.) from sequence
            sequence = re.sub(r'\[[^\]]*\]', '', sequence)
            # Convert comma-separated string to list of integers
            resi_list = list(map(int, resi.split(',')))
            resi_start = min(resi_list)
            resi_end = max(resi_list)
            rows.append({
                'uniprot_accn': uniprot_accn,
                'resi_start': resi_start,
                'resi_end': resi_end,
                'sequence': sequence
            })
        return pd.DataFrame(rows, columns=['uniprot_accn', 'resi_start', 'resi_end', 'sequence'])
    
    # Get old domain boundaries
    df_old_domain_boundaries = get_domain_boundaries(uniprot_accn, df_old_domainome_info)

    # Get full sequence from old domainome info
    full_sequence_row = df_old_domainome_info[df_old_domainome_info['uniprot_accn'] == uniprot_accn]
    if not full_sequence_row.empty and hasattr(full_sequence_row.iloc[0], "full_sequence"):
        full_sequence = full_sequence_row.iloc[0].full_sequence
    else:
        full_sequence = None

    # Add uniprot_accn to df_domain if not present
    if 'uniprot_accn' not in df_domain.columns:
        df_domain['uniprot_accn'] = df_domain['structure'].str.split('-').str[1]

    # Filter df_domain to rows whose structure contains the keyword (uniprot_accn)
    df_new_domain_boundaries = df_domain[df_domain['structure'].str.contains(uniprot_accn)]

    # Filter to true domains (i.e. domain > 0)
    df_new_domain_boundaries = df_new_domain_boundaries[df_new_domain_boundaries['domain'] > 0]
    # Select new boundary cols
    df_new_domain_boundaries = df_new_domain_boundaries[['uniprot_accn', 'resi_start', 'resi_end', 'sequence']].copy()

    # Sort by resi_start
    df_new_domain_boundaries = df_new_domain_boundaries.sort_values(by='resi_start')

    # Get the path to the PDB file(s)
    import subprocess
    import sys
    if hasattr(sys, 'ps1'):  # If interactive (notebook) use !
        pdb_path = !find {pdb_dir} -name "*{uniprot_accn}*.pdb"
    else:  # If script, use subprocess
        proc = subprocess.run(['find', pdb_dir, '-name', f"*{uniprot_accn}*.pdb"], capture_output=True, text=True)
        pdb_path = proc.stdout.splitlines()

    # Visualize domain boundaries
    view = show_domain_comparison_pdb(
        pdb_path=pdb_path,
        df_old_domains=df_old_domain_boundaries,
        df_new_domains=df_new_domain_boundaries
    )
    return view, df_old_domain_boundaries, df_new_domain_boundaries, full_sequence, pdb_path



In [263]:
# Check if the .pdb file exists in the old domainome input pdb directory

import os
import glob

def pdb_exists_for_uniprot(uniprot_accn, pdb_dir="/work/lpdi/users/shxiao/masif_seed/masif/data/masif_human_proteome_domains_merged/input_pdbs/compressed/"):
    pattern = os.path.join(pdb_dir, f"*{uniprot_accn}*.pdb")
    matches = glob.glob(pattern)
    return len(matches) > 0

pdb_exists = pdb_exists_for_uniprot(uniprot_accn)
pdb_exists

False

In [ ]:
# Initialize the list of uniprot accessions to visualize
uniprot_avail = df_domain['uniprot_accn'].unique()
idx = 0

array(['Q13002', 'X6R907', 'P35408', 'C9JL63'], dtype=object)

In [280]:
# Iterate through the next accession
idx = idx + 1
uniprot_accn = uniprot_accn = uniprot_avail[idx]

view, df_old_domain_boundaries, df_new_domain_boundaries, full_sequence, pdb_path = prepare_and_show_domain_comparison(
    uniprot_accn=uniprot_accn,
    df_old_domainome_info=df_old_domainome_info,
    df_domain=df_domain
)
print(f"uniprot_accn: {uniprot_accn}")
print(f"Model exists in old domainome input pdb directory: {pdb_exists_for_uniprot(uniprot_accn)}")
view

uniprot_accn: A0A3B3IU54
Model exists in old domainome input pdb directory: False


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Notable examples:
- Q9BZH6 could be split further, but PAE has low contrast

In [260]:
df_domain[df_domain['uniprot_accn'] == uniprot_accn]

,structure,domain,n_residues,resi_start,resi_end,mean_pae,sequence,deeptmhmm_annotation,uniprot_accn
0,AF-A0A0A0MT38-F1-model_v6.pdb,1,46,1,46,13.629909,MSQPPLLPASAETRKFTRALSKPGTAAELRQSVSEVVRGSVLLAKP,None,A0A0A0MT38
1,AF-A0A0A0MT38-F1-model_v6.pdb,-1,96,47,142,7.081125,KLIEPLDYENVIVQKKTQILNDCLREMLLFPYDDFQTAILRRQGRY...,None,A0A0A0MT38
2,AF-A0A0A0MT38-F1-model_v6.pdb,2,153,143,295,12.325368,VKLDKLPVHVYEVDEEVDKDEDAASLGSQKGGITKHGWLYKGNMNS...,None,A0A0A0MT38
3,AF-A0A0A0MT38-F1-model_v6.pdb,3,40,296,335,13.635174,SHEDDEQSKLEGSGSGLDSYLPELAKSAREAEIKLKSESR,None,A0A0A0MT38
4,AF-A0A0A0MT38-F1-model_v6.pdb,-2,919,336,1254,12.697976,VKLFYLDPDAQKLDFSSAEPEVKSFEEKFGKRILVKCNDLSFNLQC...,None,A0A0A0MT38


_____________

### Step 3 - Add postprocessing metrics

deepTMHMM annotation

In [25]:
import re
from Bio import pairwise2
import numpy as np
from pathlib import Path

def sequence_alignment(seqA, seqB):

    identical_reward = 2
    non_identical_penalty = -1
    gap_opening_penalty = -2
    gap_extending_penalty = -0
    # For more info run: `help(pairwise2.align.localms)`
    # aln = pairwise2.align.localms(
    aln = pairwise2.align.globalms(
        seqA, seqB, 
        match=identical_reward, 
        mismatch=non_identical_penalty, 
        open=gap_opening_penalty, 
        extend=gap_extending_penalty
    )[0]

    is_match = np.array([(a == b) and (a != '?') for a, b in zip(aln.seqA, aln.seqB)])

    existsA = np.array(list(aln.seqA)) != '-'
    matching_indsA = np.where(is_match[existsA])[0]

    existsB = np.array(list(aln.seqB)) != '-'
    matching_indsB = np.where(is_match[existsB])[0]

    return matching_indsA, matching_indsB


def parse_deeptmhmm(row, deeptmhmm_root="/work/upthomae/Meng/hp_list_DeepTMHMM/out/", placeholder='?'):
    """
    The per-residue labels are signal peptide (S), inside cell/cytosol (I),
    alpha membrane (M), beta membrane (B), periplasm (P) and outside cell/lumen 
    of ER/Golgi/lysosomes (O).
    """

    # Get the structure name from the row
    structure_name = row.structure

    # Extract the uniprot accession from the structure name (2nd value after splitting by '-')
    uniprot_accn = structure_name.split('-')[1]

    file = Path(deeptmhmm_root, uniprot_accn, 'predicted_topologies.3line')
    if not file.exists():
        # print("File not found.")
        return None
    
    with open(file, 'r') as f:
        header, sequence, annotation = f.read().strip().splitlines()
        sequence = np.array(list(sequence))
        annotation = np.array(list(annotation))

    assert len(sequence) == len(annotation)
    
    resi_start = row.resi_start
    resi_end = row.resi_end
    domain_indices = np.arange(resi_start, resi_end + 1)
    domain_sequence = re.sub(r'\[[A-Z]{3}\]', placeholder, row.sequence)
    domain_sequence = np.array(list(domain_sequence))

    domain_labels = np.full(len(domain_sequence), fill_value=placeholder)
    try:
        # Match residue numbers directly
        is_valid = domain_sequence != placeholder
        global_index = domain_indices[is_valid] - 1
        assert np.all(sequence[global_index] == domain_sequence[is_valid])   
    except (IndexError, AssertionError) as e:
        # Perform a sequence alignment
        inds_seq1, inds_seq2 = sequence_alignment(''.join(domain_sequence), ''.join(sequence))
        is_valid = inds_seq1
        global_index = inds_seq2
        # print(''.join(domain_sequence[is_valid]) in ''.join(sequence))
        # print(''.join(sequence))
        # print(''.join(domain_sequence))
        # print(global_index)
        assert np.all(sequence[global_index] == domain_sequence[is_valid])
    
    domain_labels[is_valid] = annotation[global_index]
    assert np.isin(domain_labels, ['B', 'P', 'M', 'I', 'O', 'S', placeholder]).all()  # Beta, Periplasm, Membrane, Inside, Outside, Signal, [Placeholder]

    return ''.join(domain_labels)


In [26]:
df_domain['deeptmhmm_annotation'] = df_domain.apply(parse_deeptmhmm, axis=1)

In [27]:
df_domain.head()

,structure,domain,n_residues,resi_start,resi_end,mean_pae,sequence,deeptmhmm_annotation
0,AF-A0A3B3IU53-F1-model_v6.pdb,-1,22,1,22,11.882775,MKMMKTKEEPELQTRREMEERT,None
1,AF-A0A3B3IU53-F1-model_v6.pdb,1,48,23,70,3.351020,ITIEIPEVLKKQLEDDCYYINRRKRLVKLPCQTNIITILESYVKHFAI,None
0,AF-Q13002-F1-model_v6.pdb,1,33,1,33,11.394130,MKIIFPILSNPVFRRTVKLLLCLLWIGYSQGTT,SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSOO
1,AF-Q13002-F1-model_v6.pdb,2,385,34,418,6.971705,HVLRFGGIFEYVESGPMGAEELAFRFAVNTINRNRTLLPNTTLTYD...,OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO...
2,AF-Q13002-F1-model_v6.pdb,-1,10,419,428,7.931507,GKPANITDSL,OOOOOOOOOO
